# 03 — SHAP explanation extraction (source-trained model, both datasets)

For each seed's trained model, compute per-class DeepSHAP rankings on:
- the **source** test split -> E1
- the **target** dataset -> E2 (same model, no retraining)

Rankings are pickled to `results/tables/shap_rankings_seed{seed}.pkl` for
notebook 04 to consume.

**Explainer choice matters here.** `shap.DeepExplainer` does not implement DeepLIFT
for `nn.LSTM` — it warns `unrecognized nn.Module: LSTM` and degrades to a plain
gradient for that layer, so its output is not a valid DeepSHAP value for a CNN-LSTM.
The config defaults to `explainer_type: gradient` (Expected Gradients), which handles
recurrent layers correctly. Re-run with `deep` to produce the robustness appendix.


In [ ]:
import sys, os, pickle
sys.path.append(os.path.abspath('..'))

import yaml
import numpy as np
import torch

from src.training.train import get_device
from src.models.cnn_lstm import build_model
from src.explain.shap_utils import compute_shap_rankings
from src.data.schema import SHARED_FEATURES

with open('../configs/default.yaml') as f:
    cfg = yaml.safe_load(f)
with open('../data/processed/class_names.yaml') as f:
    class_names = yaml.safe_load(f)

device = get_device(cfg['train']['device'])
source = np.load('../data/processed/cicids2017_source.npz')
target = np.load('../data/processed/unsw_nb15_target.npz')
n_features = source['X_train'].shape[1]
n_classes = len(class_names)
os.makedirs('../results/tables', exist_ok=True)

In [ ]:
for seed in cfg['seeds'][:cfg['n_seeds']]:
    print(f'\n=== seed {seed} ===')
    model = build_model(cfg, n_features, n_classes)
    model.load_state_dict(torch.load(f'../checkpoints/cnn_lstm_seed{seed}.pt', map_location=device))
    model.to(device)

    source_rankings = compute_shap_rankings(
        model, X_background=source['X_train'], X_explain=source['X_test'], y_explain=source['y_test'],
        class_names=class_names, feature_names=SHARED_FEATURES, device=device,
        background_samples=cfg['shap']['background_samples'],
        explain_samples_per_class=cfg['shap']['explain_samples_per_class'], seed=seed,
        explainer_type=cfg['shap']['explainer_type'],
    )
    target_rankings = compute_shap_rankings(
        model, X_background=source['X_train'], X_explain=target['X'], y_explain=target['y'],
        class_names=class_names, feature_names=SHARED_FEATURES, device=device,
        background_samples=cfg['shap']['background_samples'],
        explain_samples_per_class=cfg['shap']['explain_samples_per_class'], seed=seed,
        explainer_type=cfg['shap']['explainer_type'],
    )

    with open(f'../results/tables/shap_rankings_seed{seed}.pkl', 'wb') as f:
        pickle.dump({'source': source_rankings, 'target': target_rankings}, f)
    print('Saved rankings for seed', seed)